# AlphaZero Gomoku - Two-Phase Training (Optimized)

Notebook huấn luyện agent AlphaZero cho game Caro (Gomoku) 15x15.

**Thời gian ước tính:** 6-12 giờ trên Colab GPU (T4/V100)

## Tối ưu chính:
- **Dirichlet alpha=0.15** (tuned cho 225 actions: ~10/N)
- **c_puct=2.5** (khám phá mạnh hơn)
- **Phase 1:** MCTS sims=100 (nhanh hơn), 40 games/iter, label smoothing
- **Phase 2:** MCTS sims=200 (chính xác hơn), 30 games/iter
- **LR=0.001** + CosineAnnealing scheduler
- **Mixed precision (fp16)** tự động trên GPU
- **Replay buffer 100K** (giữ nhiều dữ liệu hơn)

## Quy trình 2 pha:
1. **Phase 1 - Học từ MiniMax:** depth 1→2→3→5, imitation learning
2. **Phase 2 - Self-play:** fine-tune, vượt MiniMax

## 1. Setup

In [ ]:
# Clone repo
!git clone https://github.com/VanLam05/Caro-AI.git
%cd Caro-AI

In [ ]:
# Install dependencies
!pip install torch numpy

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Cấu hình Training

### Tham số đã tối ưu:

| Tham số | Giá trị | Lý do |
|---------|---------|-------|
| `c_puct` | 2.5 | Khám phá mạnh hơn (cũ: 1.5) |
| `dirichlet_alpha` | 0.15 | Tuned cho 225 actions (~10/N, cũ: 0.3) |
| `lr` | 0.001 | Ổn định hơn (cũ: 0.002) |
| `epochs_per_iter` | 10 | Train kỹ hơn mỗi batch (cũ: 5) |
| `replay_buffer` | 100K | Nhiều dữ liệu hơn (cũ: 50K) |
| Phase 1 sims | 100 | Nhanh hơn, MiniMax là teacher |
| Phase 2 sims | 200 | Chính xác hơn cho self-play |
| Label smoothing | 0.1 | Tránh overfit vào exact MiniMax moves |

| Cấu hình | P1 Iters | P2 Iters | P1 Games | P2 Games | Thời gian |
|-----------|:---:|:---:|:---:|:---:|:---:|
| Nhanh     | 10  | 5   | 25  | 20  | ~3-4h  |
| Chuẩn     | 20  | 15  | 40  | 30  | ~8-12h |
| Kỹ        | 30  | 20  | 50  | 40  | ~18-24h|

In [ ]:
# Training configuration (optimized defaults)
CONFIG = {
    # Phase 1: Learn from MiniMax (imitation learning)
    'phase1_iterations': 20,
    'phase1_games': 40,
    'phase1_simulations': 100,   # Fewer sims, MiniMax is the teacher
    
    # Phase 2: Self-play refinement
    'phase2_iterations': 15,
    'phase2_games': 30,
    'phase2_simulations': 200,   # More sims for accurate self-play
    
    # Network
    'num_res_blocks': 5,
    'channels': 128,
    
    # Training
    'lr': 0.001,
    'batch_size': 256,
    'epochs_per_iter': 10,
    'c_puct': 2.5,
    'checkpoint_dir': 'neural_net',
}

## 3. Training

In [ ]:
import sys
sys.path.insert(0, '.')

from pipeline.train import AlphaZeroTrainer

trainer = AlphaZeroTrainer(
    phase1_iterations=CONFIG['phase1_iterations'],
    phase1_games=CONFIG['phase1_games'],
    phase1_simulations=CONFIG['phase1_simulations'],
    phase2_iterations=CONFIG['phase2_iterations'],
    phase2_games=CONFIG['phase2_games'],
    phase2_simulations=CONFIG['phase2_simulations'],
    num_res_blocks=CONFIG['num_res_blocks'],
    channels=CONFIG['channels'],
    lr=CONFIG['lr'],
    batch_size=CONFIG['batch_size'],
    epochs_per_iter=CONFIG['epochs_per_iter'],
    c_puct=CONFIG['c_puct'],
    checkpoint_dir=CONFIG['checkpoint_dir'],
)

print("Starting optimized two-phase training...")
trainer.train()

## 4. Evaluation

In [ ]:
from neural_net.architecture import GomokuNet
from mcts.mcts_alpha_zero import MCTS
from game.board import Board
from models.agentMiniMax import AgentMiniMax
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

net = GomokuNet(board_size=15, in_channels=4, num_res_blocks=5, channels=128)
net.to(device)
net.load_checkpoint('neural_net/model_checkpoint.pth', device=device)
net.eval()

mcts = MCTS(net, num_simulations=200)

for depth in [1, 3, 5]:
    wins = 0
    num_eval_games = 1
    
    for g in range(num_eval_games):
        board = Board(rows=15, cols=15, winning_condition=5)
        minimax = AgentMiniMax(board, max_depth=depth)
        rl_player = 1 if g % 2 == 0 else -1
        
        move_count = 0
        while True:
            if board.turn == rl_player:
                action, _ = mcts.get_action(board, temperature=0)
                row, col = action
            else:
                move = minimax.get_move()
                if move is None:
                    break
                row, col = move
            
            board.make_move(row, col)
            move_count += 1
            
            result = board.get_game_ended()
            if result != 0:
                if result != 0.5:
                    if (result == 1 and rl_player == board.originXO) or \
                       (result == -1 and rl_player != board.originXO):
                        wins += 1
                break
            
            if move_count >= 225:
                break
    
    print(f"vs MiniMax depth={depth}: {wins}/{num_eval_games} wins ({wins/num_eval_games:.1%})")

## 5. Download Model

In [ ]:
from google.colab import files
files.download('neural_net/model_checkpoint.pth')

In [ ]:
# Or save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('neural_net/model_checkpoint.pth', '/content/drive/MyDrive/model_checkpoint.pth')
print("Model saved to Google Drive!")

## 6. Resume Training

In [ ]:
import os

# Resume from checkpoint (skip Phase 1 if already done)
resume_trainer = AlphaZeroTrainer(
    phase1_iterations=0,
    phase2_iterations=15,
    phase2_games=30,
    phase2_simulations=200,
    lr=0.001,
    checkpoint_dir='neural_net',
)

checkpoint_path = 'neural_net/model_checkpoint.pth'
if os.path.exists(checkpoint_path):
    resume_trainer.net.load_checkpoint(checkpoint_path, device=resume_trainer.device)
    print(f"Resumed from {checkpoint_path}")

resume_trainer.train()